# 1. Basic Chatbot using LangGraph

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from dotenv import load_dotenv
import os

load_dotenv()

### Step 1: Define State Schema

In [ ]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

### Step 2: Initialize LLM

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Ensure llm is defined first before calling llm.bind_tools(tools)
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

### Step 3: Define Tools & Bind to LLM

In [ ]:
from langchain_tavily import TavilySearch

# Initialize Tavily search tool (reads TAVILY_API_KEY from .env)
tavily_tool = TavilySearch(max_results=2)
tools = [tavily_tool]

# Bind tools to the initialized llm
llm_with_tools = llm.bind_tools(tools)

### Step 4: Define Chatbot Node Function

In [ ]:
def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

### Step 5: Build & Compile Graph

In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node("llmchatbot", chatbot)
graph_builder.add_edge(START, "llmchatbot")
graph_builder.add_edge("llmchatbot", END)

graph = graph_builder.compile()

### Step 6: Invoke Graph

In [ ]:
response = graph.invoke({"messages": [HumanMessage(content="What is the weather in Tokyo?")]})
print(response["messages"][-1])